# Day 1 — Descriptive Statistics: The Mean Lied to Me

**#30DaysOfMachineLearning**

## Business problem

StreamFlix (fictional streaming platform) has 1,200 titles live on the platform.
The Content team ran a report and got worried: **"Our average content rating is only 6.45/10 — is our catalogue actually bad?"**

Before recommending the team pull titles or renegotiate licensing deals, we need to check
whether the **mean** is even the right number to trust here, or whether it's being distorted
by a small number of very low-rated titles.

**Goal of this notebook:**
1. Compute mean, median, mode, variance and standard deviation of `avg_rating`.
2. Visualize the distribution to see *why* the mean and median disagree.
3. Break the picture down by genre to find where quality is genuinely inconsistent.
4. Turn the stats into a business recommendation.


## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from stats_utils import (
    central_tendency,
    dispersion,
    skew_report,
    summarize,
    plot_distribution,
    group_dispersion,
)

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")


## 2. Load the data

Dataset generated by `generate_dataset.py` — a synthetic but realistic catalogue of 1,200 titles with a deliberate long tail of low-rated content (mirrors real streaming-platform rating distributions).

In [ ]:
df = pd.read_csv("data/streaming_ratings.csv")
print(df.shape)
df.head()


In [ ]:
df.describe(numeric_only=True)


## 3. Central tendency — mean, median, mode

This is the number the Content team already has (the mean). Let's put it next to median and mode before drawing any conclusions.

In [ ]:
ct = central_tendency(df["avg_rating"])
ct


In [ ]:
print(skew_report(df["avg_rating"]))


**Reading this:** the median sits noticeably above the mean. That's the signature of a *left-skewed* distribution — a cluster of low scores is pulling the average down, even though most titles individually score higher than the average suggests.

## 4. Visualize the distribution

Seeing *why* mean ≠ median matters more than the numbers alone.

In [ ]:
ax = plot_distribution(df["avg_rating"], title="StreamFlix catalogue — rating distribution (n=1,200)", bins=22)
plt.xlabel("Average rating")
plt.tight_layout()
plt.savefig("rating_distribution.png", dpi=150)
plt.show()


## 5. Spread — variance & standard deviation

Central tendency alone doesn't tell the team *how consistent* quality is. Two catalogues can share a mean and still tell very different stories.

In [ ]:
d = dispersion(df["avg_rating"])
d


With a standard deviation of roughly **2.4 rating points** on a 1–10 scale, ratings are fairly spread out — worth checking whether that spread is even across genres, or concentrated in a few problem areas.

## 6. Where is the inconsistency coming from? (segment by genre)

Aggregate stats can hide the real story. Breaking down by genre shows *where* to focus.

In [ ]:
genre_stats = group_dispersion(df, group_col="genre", value_col="avg_rating")
genre_stats


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
genre_stats["std_dev"].sort_values().plot(kind="barh", color="#F96167", ax=ax)
ax.set_xlabel("Standard deviation of rating")
ax.set_title("Rating consistency by genre (lower = more consistent)")
plt.tight_layout()
plt.savefig("genre_std_dev.png", dpi=150)
plt.show()


## 7. Business takeaways

- **Don't lead with the mean on skewed data.** The median (typical title) tells a materially more optimistic and more accurate story than the mean the Content team started with.
- **The real problem is a *tail*, not the whole catalogue.** A minority of very low-rated titles is dragging the average down — the fix is targeted (audit/retire the worst-performing titles) rather than platform-wide.
- **Genre breakdown points to where to act.** The genres with the highest standard deviation are the ones with the least consistent quality — that's where a content audit will have the most impact.
- **Recommendation:** report median rating (not mean) as the primary catalogue-health KPI going forward, and stand up a quarterly review of the bottom-decile titles by rating.


---
*Day 1 of #30DaysOfMachineLearning — Descriptive Statistics. Code: `stats_utils.py` · Data: `data/streaming_ratings.csv`*